In [87]:
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [89]:
df = pd.read_csv("clean_skills_dataset.csv")
df.head()

,skills,job_position_name
0,brand promotion campaign management field supe...,"Executive/ Senior Executive- Trade Marketing, ..."
1,fast typing skill ieltsinternet browsing onlin...,Business Development Executive
2,ios ios app developer ios application developm...,Senior iOS Engineer
3,python r or java tensorflow pytorch scikit learn,AI Engineer
4,maintenance and troubleshooting mechanical,Mechanical Engineer


In [91]:
df.shape

(23, 2)

In [93]:
df["job_position_name"].nunique()

23

In [95]:
df_grouped = (
    df.groupby("job_position_name")["skills"]
    .apply(lambda x: " ".join(x))
    .reset_index()
)

df_grouped.head()

,job_position_name,skills
0,AI Engineer,python r or java tensorflow pytorch scikit learn
1,Asst. Manager/ Manger (Administrative),administration health safety and environment s...
2,Business Development Executive,fast typing skill ieltsinternet browsing onlin...
3,Civil Engineer,autocad etabs microsoft office suite ms project
4,Data Engineer,azure big data data analytics etl tools power ...


In [97]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2)
)

In [99]:
job_vectors = vectorizer.fit_transform(df_grouped["skills"])

In [101]:
import os
os.makedirs("model", exist_ok=True)

joblib.dump(vectorizer, "model/vectorizer.pkl")
joblib.dump(job_vectors, "model/job_vectors.pkl")
joblib.dump(df_grouped, "model/job_data.pkl")

print("Model saved!")

Model saved!


In [107]:
def match_resume(resume_text, top_n=5):
    vectorizer = joblib.load("model/vectorizer.pkl")
    job_vectors = joblib.load("model/job_vectors.pkl")
    job_data = joblib.load("model/job_data.pkl")

    # Transform resume
    resume_vec = vectorizer.transform([resume_text])

    # Similarity
    scores = cosine_similarity(resume_vec, job_vectors)[0]

    # Top matches
    top_indices = scores.argsort()[-top_n:][::-1]

    results = []
    for i in top_indices:
        results.append({
            "job": job_data.iloc[i]["job_position_name"],
            "score": round(float(scores[i]) * 100, 2)
        })

    return results

In [105]:
resume = "python machine learning deep learning data science pandas numpy"

results = match_resume(resume)

for r in results:
    print(r)

{'job': 'Data Engineer', 'score': 31.02}
{'job': 'AI Engineer', 'score': 21.68}
{'job': 'HR Officer', 'score': 0.0}
{'job': 'Asst. Manager/ Manger (Administrative)', 'score': 0.0}
{'job': 'Business Development Executive', 'score': 0.0}
